# Interactive DEM to Matrix Converter

This notebook guides you through converting Digital Elevation Model (DEM) data from CSV format to numpy matrices suitable for beaver simulation.

**Quick Start:** Select a pre-configured region ('boston' in this case for the Rumney Marsh site) in the Configuration cell below, or choose 'custom' to use your own data.

## Workflow Overview:
1. **Configure EPSG Codes** - Verify coordinate reference systems
2. **Load CSV Data** - Import elevation data from CSV file
3. **Set Resolution** - Confirm target resolution for resampling
4. **Resample DEM** - Resample to target resolution
5. **Normalize Data** - Scale elevations to [0, 1] range
6. **Transform Coordinates** - Convert between coordinate systems if needed
7. **Preview DEM** - Visualize the processed DEM
8. **Define Region** - Set polygon corners for area of interest
9. **Apply Polygon Mask** - Mask areas outside region
10. **Set Food Cache** - Define food cache location
11. **Crop & Save** - Extract final region and save to disk

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyproj import Transformer

cwd = Path.cwd().resolve()
for parent in [cwd, *cwd.parents]:
    if (parent / 'beaversim').is_dir():
        project_root = parent
        break
else:
    raise RuntimeError("Could not locate project root containing 'beaversim'.")

project_root_str = str(project_root)
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

# Import the DEM conversion module
from data_acquisition.modules.dem_to_matrix import (
    load_csv_data,
    transform_coordinates_to_target_crs,
    resample_dem_to_target_resolution,
    normalize_elevation_data,
    apply_global_stream_threshold,
    create_polygon_mask,
    crop_to_bounds,
    transform_food_cache_coordinates,
    save_dem_outputs
)

# Import custom colormap for visualization
from beaversim.ral.backend.modules.module_colors import ColorMaps
colors = ColorMaps()

print("Modules loaded successfully")

## Configuration: Select Your Example Region

Choose between pre-configured example regions or set up your own custom region.

In [ ]:
# ============================================================================
# SELECT YOUR REGION: 'boston', or 'custom'
# ============================================================================
REGION = 'boston'  # Change this to 'boston', or 'custom'

# Pre-configured settings for example regions
REGION_CONFIGS = {
    'boston': {
        'name': 'Rumney Marsh, Boston MA',
        'csv_path': '../../data_acquisition/example_datasets/RUMNEY_MARSH_DEM/RumneyMarsh.csv',
        'epsg_geographic': 'EPSG:4326',
        'epsg_source': 'EPSG:2249',  # CSV data is in NAD83 MA Mainland (US Survey Feet)
        'epsg_target': 'EPSG:6483',  # Convert to NAD83(2011) MA Mainland (meters)
        'polygon_corners': [
            (-71.0033967, 42.4345814),  # NW
            (-71.0034071, 42.4328122),  # SW
            (-71.0010287, 42.4328093),  # SE
            (-71.0009986, 42.4345665)   # NE
        ],
        'food_cache_latlon': None,
        'target_resolution_m': 1.0,  # In meters (after conversion)
        'output_dir': '../../output/RumneyMarsh_Boston/DEM'
    },    
    'custom': {
        'name': 'Custom Region',
        'csv_path': './path/to/your/data.csv',  # UPDATE THIS
        'epsg_geographic': 'EPSG:4326',
        'epsg_source': 'EPSG:6483',  # UPDATE THIS based on your CSV data
        'epsg_target': 'EPSG:6483',  # UPDATE THIS for desired output units
        'polygon_corners': [],  # Empty list = use entire map
        'food_cache_latlon': (44.365899, -68.272928),  # UPDATE THIS
        'target_resolution_m': 1.5,
        'output_dir': '../../output/CustomRegion/DEM'
    }
}

# Load the selected configuration
config = REGION_CONFIGS[REGION]

print(f"  Configuration loaded: {config['name']}")
print(f"  CSV: {config['csv_path']}")
print(f"  Source CRS (CSV): {config['epsg_source']}")
print(f"  Target CRS (Output): {config['epsg_target']}")
if config['epsg_source'] != config['epsg_target']:
    print(f"    Will convert from source to target CRS")
print(f"  Polygon corners: {len(config['polygon_corners'])} defined")
print(f"  Output directory: {config['output_dir']}")
print(f"\nNote: You can modify individual parameters in the cells below if needed.")

## Step 1: Configure EPSG Coordinate Reference Systems

### What are EPSG codes?

EPSG codes define **coordinate reference systems** (CRS) - standardized methods for specifying locations on Earth. This workflow requires two types:

**Geographic CRS** (`epsg_geographic`)
- Uses latitude/longitude in degrees (like GPS coordinates)
- Works globally but cannot directly measure distances
- Standard: `EPSG:4326` ([WGS84](https://en.wikipedia.org/wiki/World_Geodetic_System) - the GPS coordinate system)

**Projected CRS** (`epsg_projected`)
- Uses x/y coordinates in meters (or feet) for a specific region
- Enables accurate distance and area calculations
- Region-specific codes (e.g., `EPSG:6483` for Massachusetts - in meters)

### Why both?

DEM data typically comes in geographic coordinates (lat/lon), but spatial analysis requires projected coordinates (meters). This tool automatically converts between them using the EPSG codes you specify.

### Selecting your EPSG codes:

**Geographic CRS**: Use `EPSG:4326` (standard for most DEM data)

**Projected CRS**: Choose based on your study area
- **Massachusetts**: `EPSG:6483` (NAD83 Massachusetts Mainland, **meters**) or `EPSG:2249` (NAD83 Massachusetts Mainland, **US Survey Feet**)
- **Maine**: `EPSG:6484` (NAD83 Maine East)
- **California (North)**: `EPSG:6418` (NAD83 California zone 1)
- **Other regions**: Search at [epsg.io](https://epsg.io/) using your location

### ⚠️ CRITICAL: Check if your CSV uses Feet or Meters!

Many US state plane coordinate systems have **two versions**:
- One in **meters** (e.g., EPSG:26986)
- One in **US Survey Feet** (e.g., EPSG:2249)

**How to check your CSV data:**
1. Look at the coordinate values: if X/Y are ~200,000-300,000 → likely meters; if ~700,000-900,000 → likely feet
2. Check the data source documentation
3. The tool will auto-detect if coordinates are projected vs. geographic, but you must specify the **correct units**

**Example for Massachusetts:**
- `EPSG:2249` - NAD83 / Massachusetts Mainland (US Survey Feet) ← Use this if your data is in feet
- `EPSG:26986` - NAD83 / Massachusetts Mainland (meters) ← Use this if your data is in meters  
- `EPSG:6483` - NAD83(2011) / Massachusetts Mainland (meters) ← Modern version in meters

**Tip**: If your polygon mask shows 0 points inside, you likely have a units mismatch!

In [ ]:
# Load EPSG codes from configuration (can be modified if needed)
epsg_geographic = config['epsg_geographic']
epsg_source = config['epsg_source']  # CRS of the CSV data
epsg_target = config['epsg_target']  # Desired output CRS

print(f"EPSG Geographic CRS: {epsg_geographic}")
print(f"EPSG Source CRS (CSV data): {epsg_source}")
print(f"EPSG Target CRS (Output): {epsg_target}")
print(f"Region: {config['name']}")

if epsg_source != epsg_target:
    print(f"\n Note: Coordinates will be converted from {epsg_source} to {epsg_target}")

## Step 2: Load CSV Data

Specify the path to your DEM CSV file. The CSV should contain columns for coordinates (x, y or lat, lon) and elevation.

**Supported formats:**
- Geographic coordinates: `x` (longitude), `y` (latitude), `elevation`
- Projected coordinates: `X_m`, `Y_m`, `elevation`

In [ ]:
# Load CSV path from configuration (can be modified if needed)
csv_path = config['csv_path']

# Load the data using the module function
df, coordinates_are_projected = load_csv_data(csv_path)

print(f"\nData loaded: {len(df)} elevation points")
print(f"  Coordinate type: {'Projected' if coordinates_are_projected else 'Geographic'}")

## Step 3: Set Target Resolution

Define the target resolution in meters for the output grid. Lower values create higher resolution but larger matrices.

**Recommended values:**
- High detail: 0.5 - 1.0 m
- Medium detail: 1.0 - 2.0 m
- Low detail: 2.0 - 5.0 m

In [ ]:
# Load resolution from configuration (can be modified if needed)
target_resolution_m = config['target_resolution_m']
interpolation_method = 'linear'  # Options: 'linear', 'nearest', 'cubic'
# Adjust based on your data (0 to skip, 5-15 typical)
print(f"Target resolution: {target_resolution_m}m per pixel")
print(f"Interpolation method: {interpolation_method}")

## Step 4: Resample DEM Data

Resample the DEM to your target resolution.

In [ ]:
# Resample the data
X, Y, Z = resample_dem_to_target_resolution(df, target_resolution_m, interpolation_method)

print(f"\n  Resampled data shape: {Z.shape}")
print(f"  Grid dimensions: {Z.shape[0]} x {Z.shape[1]} pixels")
print(f"  Approximate area: {(X.max()-X.min())*(Y.max()-Y.min()):.2f} m²")

## Step 5: Normalize Elevation Data

Normalize the raw elevation data to [0, 1] range for land, marking water/NaN as invalid.

In [ ]:
# Normalize elevation data to [0, 1] for land
Z = normalize_elevation_data(Z)

print(f"\n  Normalized elevation data")
print(f"  Land values now in range [0, 1]")
print(f"  Water/invalid marked for later processing")

## Step 6: Transform Coordinates (if needed)

If your CSV data is in feet but you want output in meters (or vice versa), transform the coordinates now.

In [ ]:
# Transform coordinates from source CRS to target CRS (e.g., feet → meters)
if epsg_source != epsg_target:
    df, X, Y = transform_coordinates_to_target_crs(df, X, Y, epsg_source, epsg_target)
    print(f"\n  Coordinates transformed from {epsg_source} to {epsg_target}")
else:
    print(f"Source and target CRS are the same - no transformation needed")

## Step 7: Preview Full DEM

Visualize the entire DEM (with streams removed) to help you select the region of interest.

In [ ]:
plt.figure(figsize=(10, 8))
plt.imshow(Z, extent=[X.min(), X.max(), Y.min(), Y.max()], origin='lower', cmap='Grays')
cbar = plt.colorbar(shrink=0.9)
cbar.ax.tick_params(labelsize=16)
cbar.ax.set_ylabel('Normalized Elevation', fontsize=20)
plt.title('Full DEM with Streams Removed - Select Your Region')
plt.xlabel('X (m)' if coordinates_are_projected else 'Longitude')
plt.ylabel('Y (m)' if coordinates_are_projected else 'Latitude')
plt.grid(True, alpha=0.3)
plt.show()

print("\nUse this visualization to identify corner coordinates for your region of interest.")

## Step 9: Define Region of Interest (Polygon)

Define the polygon corners for your area of interest. If you want to use the entire map, set `polygon_corners = []`.

**Format:** List of (longitude, latitude) tuples for geographic coordinates, or (x, y) for projected.

**Order:** Define corners in clockwise or counter-clockwise order (typically: NW, SW, SE, NE).

In [ ]:
# Load polygon corners from configuration (can be modified if needed)
polygon_corners = config['polygon_corners']

print(f"Region: {config['name']}")
if len(polygon_corners) > 0:
    print(f"Polygon defined with {len(polygon_corners)} corners:")
    for i, (lon, lat) in enumerate(polygon_corners):
        print(f"  Corner {i+1}: ({lon:.6f}, {lat:.6f})")
else:
    print("Using entire map (no polygon restriction)")
    
print(f"\nNote: You can modify polygon_corners above to define a custom region.")

## Step 10: Create and Apply Polygon Mask

Transform the coordinates to the projected system, create a mask for the selected region, and apply it to mark areas outside as water.

In [ ]:
# Use the module function to create polygon mask
# Note: After coordinate transformation, we use epsg_target for the polygon mask
inside_grid, (xmin, xmax, ymin, ymax), df = create_polygon_mask(
    X, Y, polygon_corners, coordinates_are_projected, 
    epsg_geographic, epsg_target, df
)

print(f"Points inside region: {np.sum(inside_grid)} / {inside_grid.size}")
print(f"Polygon bounds: x[{xmin:.2f}, {xmax:.2f}], y[{ymin:.2f}, {ymax:.2f}]")
print(f"DEM extent: x[{X.min():.2f}, {X.max():.2f}], y[{Y.min():.2f}, {Y.max():.2f}]")

# Visualize the DEM with polygon boundary
plt.figure(figsize=(10, 8))
plt.imshow(Z, extent=[X.min(), X.max(), Y.min(), Y.max()], origin='lower', cmap=colors._bluebrowngreen_colormap)
plt.title('DEM with Selected Region Masked')
cbar = plt.colorbar(shrink=0.8)
cbar.ax.tick_params(labelsize=16)
cbar.ax.set_ylabel('Normalized Elevation', fontsize=20)
plt.xlabel('X (m)', fontsize=20)
plt.ylabel('Y (m)', fontsize=20)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)

# Draw a box around the polygon region
if len(polygon_corners) > 0:
    from matplotlib.patches import Rectangle
    rect = Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, 
                     linewidth=3, edgecolor='red', facecolor='none', label='Selected Region')
    plt.gca().add_patch(rect)
    plt.legend(loc='upper right')
    
    # Add corner markers for debugging
    plt.plot([xmin, xmax, xmax, xmin, xmin], 
             [ymin, ymin, ymax, ymax, ymin], 
             'ro-', markersize=8, linewidth=2, label='Polygon Corners')

plt.tight_layout()
plt.show()

## Step 11: Set Food Cache Location

Define the food cache coordinates (latitude, longitude).

In [ ]:
# Load food cache location from configuration (can be modified if needed)
food_cache_latlon = config['food_cache_latlon']

# Transform to target projected coordinates
if food_cache_latlon is not None:
    tf = Transformer.from_crs(epsg_geographic, epsg_target, always_xy=True)
    food_cache_x, food_cache_y = tf.transform(food_cache_latlon[1], food_cache_latlon[0])

    print(f"Food cache coordinates:")
    print(f"  Geographic: {food_cache_latlon}")
    print(f"  Projected ({epsg_target}): ({food_cache_x:.2f}, {food_cache_y:.2f})")

## Step 12: Crop to Region Bounds and Save

Extract the final cropped region and save all processed data to disk.

In [ ]:
# Crop to region bounds
Z_cropped, X_clipped, Y_clipped, x_clipped, y_clipped = crop_to_bounds(
    Z, X, Y, (xmin, xmax, ymin, ymax)
)

print(f"  Cropped to region bounds")
print(f"  Shape: {Z_cropped.shape}")
print(f"  X range: [{x_clipped.min():.2f}, {x_clipped.max():.2f}]")
print(f"  Y range: [{y_clipped.min():.2f}, {y_clipped.max():.2f}]")

# Normalize cropped elevation data again to [0, 1]
Z_cropped = normalize_elevation_data(Z_cropped)

# Load output directory from configuration
output_dir = config['output_dir']

# Transform food cache coordinates and find indices
food_cache_proj, food_cache_idx, food_cache_in_bounds = transform_food_cache_coordinates(
    food_cache_latlon, x_clipped, y_clipped,
    epsg_geographic, epsg_target
)

# Save all outputs
save_dem_outputs(
    output_dir, Z_cropped, X_clipped, Y_clipped,
    x_clipped, y_clipped, target_resolution_m, interpolation_method,
    food_cache_latlon, food_cache_proj, food_cache_idx, food_cache_in_bounds,
    epsg_target
)

print(f"\n  Processing complete!")
print(f"  Matrix shape: {Z_cropped.shape}")
print(f"  Resolution: {target_resolution_m}m per pixel")
print(f"  Total area: {(x_clipped.max() - x_clipped.min()) * (y_clipped.max() - y_clipped.min()):.2f} m²")
print(f"  Food cache in bounds: {food_cache_in_bounds}")
print(f"  Output CRS: {epsg_target}")
print(f"  Output directory: {output_dir}")

## Final Visualization

Visualize the final processed elevation map.

In [ ]:
# Visualize the final processed elevation map
plt.figure(figsize=(10, 8))

# Calculate extent from coordinates
extent = [
    x_clipped.min(),
    x_clipped.max(),
    y_clipped.min(),
    y_clipped.max()
]

print(f"Z_cropped shape: {Z_cropped.shape}")
print(f"Extent: [{extent[0]:.2f}, {extent[1]:.2f}] x [{extent[2]:.2f}, {extent[3]:.2f}]")

plt.imshow(Z_cropped, extent=extent,
           origin='lower', cmap=colors._bluebrowngreen_colormap, aspect='equal', vmin=-1, vmax=1)
cbar = plt.colorbar(shrink=0.9)
cbar.ax.tick_params(labelsize=16)
cbar.ax.set_ylabel('Normalized Elevation', fontsize=20)
# Mark food cache location if in bounds
if food_cache_in_bounds:
    plt.plot(food_cache_proj[0], food_cache_proj[1], 'r*', markersize=15, 
             label='Food Cache', markeredgecolor='white', markeredgewidth=1)
    plt.legend(fontsize=16)

plt.title(f'Final Processed DEM -- {Z_cropped.shape[0]}x{Z_cropped.shape[1]} @ {target_resolution_m}m resolution')
plt.xlabel('X (m)', fontsize=20)
plt.ylabel('Y (m)', fontsize=20)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n  Conversion complete!")
print(f"  Files saved: elevation.npy, X_coordinates.npy, Y_coordinates.npy, processing_metadata.json")